## Linear Classifier in TensorFlow 
Using Low Level API in Eager Execution mode

### Load tensorflow

In [0]:
import tensorflow as tf

In [0]:
#Enable Eager Execution if using tensflow version < 2.0
#From tensorflow v2.0 onwards, Eager Execution will be enabled by default


### Collect Data

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [0]:
import pandas as pd

In [0]:
data = pd.read_csv('/content/drive/My Drive/gllab/prices.csv')

In [5]:
data.head()

,date,symbol,open,close,low,high,volume
0,2016-01-05 00:00:00,WLTW,123.430000,125.839996,122.309998,126.250000,2163600.0
1,2016-01-06 00:00:00,WLTW,125.239998,119.980003,119.940002,125.540001,2386400.0
2,2016-01-07 00:00:00,WLTW,116.379997,114.949997,114.930000,119.739998,2489500.0
3,2016-01-08 00:00:00,WLTW,115.480003,116.620003,113.500000,117.440002,2006300.0
4,2016-01-11 00:00:00,WLTW,117.010002,114.970001,114.089996,117.330002,1408600.0


### Check all columns in the dataset

In [6]:
data.columns

Index(['date', 'symbol', 'open', 'close', 'low', 'high', 'volume'], dtype='object')

### Drop columns `date` and  `symbol`

In [7]:
new_df=data.drop(['date','symbol'],axis =1)
new_df.head(5)

,open,close,low,high,volume
0,123.430000,125.839996,122.309998,126.250000,2163600.0
1,125.239998,119.980003,119.940002,125.540001,2386400.0
2,116.379997,114.949997,114.930000,119.739998,2489500.0
3,115.480003,116.620003,113.500000,117.440002,2006300.0
4,117.010002,114.970001,114.089996,117.330002,1408600.0


### Consider only first 1000 rows in the dataset for building feature set and target set
Target 'Volume' has very high values. Divide 'Volume' by 1000,000

In [8]:
new_df['volume']=new_df['volume']/1000
new_df.head(5)
#divided by 1000
target_df=new_df.head(1000)
target_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 5 columns):
open      1000 non-null float64
close     1000 non-null float64
low       1000 non-null float64
high      1000 non-null float64
volume    1000 non-null float64
dtypes: float64(5)
memory usage: 39.1 KB


In [9]:
target_df.describe().transpose()

,count,mean,std,min,25%,50%,75%,max
open,1000.0,66.655732,58.409247,1.53,29.2200,48.184999,113.412498,627.181073
close,1000.0,66.776202,58.360379,1.61,29.1800,48.115002,113.340001,626.751061
low,1000.0,65.979942,57.908642,1.51,28.9125,47.550002,111.655001,624.241073
high,1000.0,67.322752,58.732014,1.61,29.4750,48.500000,114.909998,629.511067
volume,1000.0,5314.222000,14465.649534,10.00,821.6250,2064.450000,4620.475000,215620.200000


### Divide the data into train and test sets

In [0]:
from sklearn.model_selection import train_test_split
x=target_df.drop('volume',axis =1)
y=target_df['volume']

#### Convert Training and Test Data to numpy float32 arrays


In [0]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.3,random_state=7)
import numpy as np
x_train=np.array(x_train)
y_train=np.array(y_train)
x_test=np.array(x_test)
y_test=np.array(y_test)

### Normalize the data
You can use Normalizer from sklearn.preprocessing

In [0]:
#Input features
x = tf.placeholder(shape=[None,4],dtype=tf.float32, name='x-input')

#Normalize the data
x_n = tf.nn.l2_normalize(x,1)

#Actual Prices
y_ = tf.placeholder(shape=[None],dtype=tf.float32, name='y-input')

## Building the Model in tensorflow

1.Define Weights and Bias, use tf.zeros to initialize weights and Bias

In [0]:
W = tf.Variable(tf.zeros(shape=[4,1]), name="Weights")
b = tf.Variable(tf.zeros(shape=[1]),name="Bias")

2.Define a function to calculate prediction

In [0]:
y = tf.add(tf.matmul(x_n,W),b,name='output')

3.Loss (Cost) Function [Mean square error]

In [0]:
loss = tf.reduce_mean(tf.square(y-y_),name='Loss')

4.Function to train the Model

1.   Record all the mathematical steps to calculate Loss
2.   Calculate Gradients of Loss w.r.t weights and bias
3.   Update Weights and Bias based on gradients and learning rate to minimize loss

In [0]:
train_op = tf.train.GradientDescentOptimizer(0.03).minimize(loss)

## Train the model for 100 epochs 
1. Observe the training loss at every iteration
2. Observe Train loss at every 5th iteration

In [0]:
#Lets start graph Execution
sess = tf.Session()

# variables need to be initialized before we can use them
sess.run(tf.global_variables_initializer())

#how many times data need to be shown to model
training_epochs = 100

### Get the shapes and values of W and b

In [19]:
for epoch in range(training_epochs):
            
    #Calculate train_op and loss
    _, train_loss = sess.run([train_op,loss],feed_dict={x:x_train, y_:y_train})
    
    if epoch % 10 == 0:
        print ('Training loss at step: ', epoch, ' is ', train_loss)

Training loss at step:  0  is  305429060.0
Training loss at step:  10  is  276395550.0
Training loss at step:  20  is  274143520.0
Training loss at step:  30  is  273968830.0
Training loss at step:  40  is  273955260.0
Training loss at step:  50  is  273954240.0
Training loss at step:  60  is  273954140.0
Training loss at step:  70  is  273954140.0
Training loss at step:  80  is  273954140.0
Training loss at step:  90  is  273954140.0


### Model Prediction on 1st Examples in Test Dataset

## Classification using tf.Keras

In this exercise, we will build a Deep Neural Network using tf.Keras. We will use Iris Dataset for this exercise.

### Load the given Iris data using pandas (Iris.csv)

In [0]:
import pandas as pd
df=pd.read_csv('/content/drive/My Drive/gllab/Iris.csv')
df=df.drop('Id',axis=1)


### Target set has different categories. So, Label encode them. And convert into one-hot vectors using get_dummies in pandas.

In [0]:
df.head(5)
new_df = pd.get_dummies(df,prefix=['Species'], columns=['Species'])

In [34]:
new_df.head()
new_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 7 columns):
SepalLengthCm              150 non-null float64
SepalWidthCm               150 non-null float64
PetalLengthCm              150 non-null float64
PetalWidthCm               150 non-null float64
Species_Iris-setosa        150 non-null uint8
Species_Iris-versicolor    150 non-null uint8
Species_Iris-virginica     150 non-null uint8
dtypes: float64(4), uint8(3)
memory usage: 5.2 KB


### Splitting the data into feature set and target set

In [30]:
from keras.models import Sequential
from keras.layers import Dense


Using TensorFlow backend.


In [0]:
from sklearn.model_selection import train_test_split

X =  new_df.drop(["Species_Iris-setosa",'Species_Iris-versicolor',
       'Species_Iris-virginica'], axis=1)
y =  new_df[['Species_Iris-setosa', 'Species_Iris-versicolor',
       'Species_Iris-virginica']]
Xtrain,Xtest,ytrain,ytest = train_test_split(X, y, test_size=0.30, random_state=1)

###  Building Model in tf.keras

Build a Linear Classifier model  <br>
1.  Use Dense Layer  with input shape of 4 (according to the feature set) and number of outputs set to 3<br> 
2. Apply Softmax on Dense Layer outputs <br>
3. Use SGD as Optimizer
4. Use categorical_crossentropy as loss function 

In [36]:
from sklearn.neural_network import MLPClassifier
model=MLPClassifier(hidden_layer_sizes=(4))
model.fit(Xtrain,ytrain)

/usr/local/lib/python3.6/dist-packages/sklearn/neural_network/multilayer_perceptron.py:566: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  % self.max_iter, ConvergenceWarning)


MLPClassifier(activation='relu', alpha=0.0001, batch_size='auto', beta_1=0.9,
              beta_2=0.999, early_stopping=False, epsilon=1e-08,
              hidden_layer_sizes=4, learning_rate='constant',
              learning_rate_init=0.001, max_iter=200, momentum=0.9,
              n_iter_no_change=10, nesterovs_momentum=True, power_t=0.5,
              random_state=None, shuffle=True, solver='adam', tol=0.0001,
              validation_fraction=0.1, verbose=False, warm_start=False)

### Model Training 

In [37]:
from keras.models import Sequential
from keras.layers import Dense

model = Sequential()
model.add(Dense(3, input_dim=4, kernel_initializer='normal', activation='softmax'))
#model.add(Dense(3, kernel_initializer='normal'))
# Compile model
model.compile(loss='categorical_crossentropy', optimizer='SGD', metrics= ['accuracy'])

W0721 11:49:03.172497 139975806023552 deprecation_wrapper.py:119] From /usr/local/lib/python3.6/dist-packages/keras/backend/tensorflow_backend.py:74: The name tf.get_default_graph is deprecated. Please use tf.compat.v1.get_default_graph instead.

W0721 11:49:03.175612 139975806023552 deprecation_wrapper.py:119] From /usr/local/lib/python3.6/dist-packages/keras/backend/tensorflow_backend.py:517: The name tf.placeholder is deprecated. Please use tf.compat.v1.placeholder instead.

W0721 11:49:03.184096 139975806023552 deprecation_wrapper.py:119] From /usr/local/lib/python3.6/dist-packages/keras/backend/tensorflow_backend.py:4115: The name tf.random_normal is deprecated. Please use tf.random.normal instead.

W0721 11:49:03.200064 139975806023552 deprecation_wrapper.py:119] From /usr/local/lib/python3.6/dist-packages/keras/optimizers.py:790: The name tf.train.Optimizer is deprecated. Please use tf.compat.v1.train.Optimizer instead.

W0721 11:49:03.220213 139975806023552 deprecation_wrapper.

### Model Prediction

In [41]:
model.fit(Xtrain, ytrain, validation_data=(Xtest,ytest), epochs=10, batch_size=5)
# evaluate the model
scores = model.evaluate(X, y)
scores[1]*100

Train on 105 samples, validate on 45 samples
Epoch 1/10
105/105 [==============================] - 0s 590us/step - loss: 0.5661 - acc: 0.7714 - val_loss: 0.6312 - val_acc: 0.6000
Epoch 2/10
105/105 [==============================] - 0s 505us/step - loss: 0.5510 - acc: 0.7524 - val_loss: 0.6019 - val_acc: 0.6000
Epoch 3/10
105/105 [==============================] - 0s 514us/step - loss: 0.5407 - acc: 0.7905 - val_loss: 0.5927 - val_acc: 0.6000
Epoch 4/10
105/105 [==============================] - 0s 522us/step - loss: 0.5301 - acc: 0.7524 - val_loss: 0.5518 - val_acc: 0.6222
Epoch 5/10
105/105 [==============================] - 0s 521us/step - loss: 0.5246 - acc: 0.7238 - val_loss: 0.5202 - val_acc: 0.9556
Epoch 6/10
105/105 [==============================] - 0s 525us/step - loss: 0.4919 - acc: 0.8000 - val_loss: 0.5091 - val_acc: 0.9111
Epoch 7/10
105/105 [==============================] - 0s 501us/step - loss: 0.4995 - acc: 0.8571 - val_loss: 0.5704 - val_acc: 0.6000
Epoch 8/10
105/10

68.0

### Save the Model

In [0]:
model.save("model.dump")

### Build and Train a Deep Neural network with 2 hidden layer  - Optional - For Practice

Does it perform better than Linear Classifier? What could be the reason for difference in performance?